+ 심리학/인지언어학에서 점화효과.
+ 딥러닝 계층을 깊게 쌓는 것과 비슷하다.
+ 깊게 쌓을 수록 더 추상적인 생각. 그렇다면 얼마나 깊게 쌓아야하는가?
+ 너무 깊게 쌓으면 사람이 딴 생각을 하듯, 딥러닝 네트워크 역시 다른 생각으로 빠질 수도 있을 것 같다.

<br>

> 문맥의 문제!! 단어의 모호성/중의성 등을 이미지에 기반한 priming effect로 해결할 수 있지 않을까???

In [1]:
import os
os.sys.path.append('../../official_github/')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from multi_layer_net_extend import MultiLayerNetExtend
from common.optimizer import SGD, Adam

In [3]:
test_dict = {}
train_dict = {}
root_path = '../datasets/cifar-10-batches-py/'


def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

accepted_keys = [b'labels', b'data'] ## select keys from [b'batch_label', b'labels', b'data', b'filenames']
for dir in os.listdir(root_path):
    if dir == 'batches.meta':
        pass
    elif dir == 'test_batch':
        test_batch = unpickle(os.path.join(root_path, dir))
        test_dict.update((k,np.array(v)) for k,v in test_batch.items() if k in accepted_keys)
    else:
        train_batch = unpickle(os.path.join(root_path, dir))
        if not train_dict:
            train_dict.update((k,np.array(v)) for k,v in train_batch.items() if k in accepted_keys)
        else:
            for key in accepted_keys:
                train_dict[key] = np.concatenate((train_dict[key], train_batch[key]), axis=0)


## CIFAR data's shape are (N, H*W*C). 
## If you want to convert them into numpy-standard (N, H, W, C) shape, run following codes.
def numpy2rgb(arr: np.ndarray) -> np.ndarray:
    batch_size = len(arr)
    arr = arr.reshape(batch_size, 3, 32, 32).transpose(0,2,3,1)
    
    return arr


train_dict[b'data'] = numpy2rgb(train_dict[b'data'])
test_dict[b'data'] = numpy2rgb(test_dict[b'data'])

In [4]:
def normalize(data: np.ndarray, mean: int=None, std: int=None) -> np.ndarray:
    if mean == None:
        mean = np.mean(data)
        
    if std == None:
        std = np.std(data)
    
    data = (data - mean) / std
    return data

In [10]:
x_train, t_train = normalize(train_dict[b'data']), train_dict[b'labels']
x_test, t_test = normalize(test_dict[b'data']), test_dict[b'labels']

print(x_train.shape, x_train.dtype)
print(np.max(x_train))
print(t_train.shape, t_train.dtype)

(50000, 32, 32, 3) float64
2.09341038199596
(50000,) int32


In [6]:
max_epochs = 20
train_size = x_train.shape[0]
batch_size = 100

iters = 5_000
iter_per_epoch = max(train_size / batch_size, 1)

learning_rate = 0.01

network_dict = {
                'Affine_2': MultiLayerNetExtend(input_size=3072, hidden_size_list=[100], output_size=10, activation='leaky', weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                'Affine_3': MultiLayerNetExtend(input_size=3072, hidden_size_list=[100, 100], output_size=10, activation='leaky', weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                'Affine_4': MultiLayerNetExtend(input_size=3072, hidden_size_list=[100, 100, 100], output_size=10, activation='leaky', weight_init_std='relu', weight_decay_lambda=0.1, use_batchnorm=True, use_dropout=True),
                }

output_dict = {}

for key, network in network_dict.items():
    net = network
    optimizer = Adam(lr=learning_rate)

    output_dict[f'{key} train acc'] = []
    output_dict[f'{key} test acc'] = []
    
    print(f'=== {key} network training start ===')
    for i in tqdm(range(iters), leave=False):
        batch_mask = np.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        t_batch = t_train[batch_mask]

        grads = net.gradient(x_batch, t_batch)
        optimizer.update(net.params, grads)

        if i % iter_per_epoch == 0:
            train_acc = net.accuracy(x_train, t_train)
            test_acc = net.accuracy(x_test, t_test)
            output_dict[f'{key} train acc'].append(train_acc)
            output_dict[f'{key} test acc'].append(test_acc)
            print(f'=== {key} result report ===')
            print(f'train_acc: {train_acc}, test_acc: {test_acc}')
    
    

=== Affine_2 network training start ===


  0%|          | 7/5000 [00:01<14:30,  5.74it/s]  

=== Affine_2 result report ===
train_acc: 0.1812, test_acc: 0.1819


 10%|█         | 506/5000 [00:10<05:52, 12.74it/s]

=== Affine_2 result report ===
train_acc: 0.2313, test_acc: 0.2306


...

=== Affine_4 result report ===
train_acc: 0.1678, test_acc: 0.1654


 80%|████████  | 4007/5000 [01:14<00:59, 16.70it/s]

=== Affine_4 result report ===
train_acc: 0.15884, test_acc: 0.1592


 90%|█████████ | 4507/5000 [01:23<00:40, 12.16it/s]

=== Affine_4 result report ===
train_acc: 0.16266, test_acc: 0.1638


In [7]:
network_dict['Affine_2'].params

{'W1': array([[-0.0014671 ,  0.01046512,  0.00058603, ...,  0.00380842,
          0.00230386,  0.00246038],
        [-0.00039465,  0.01075093,  0.00116656, ...,  0.00559324,
          0.00409153,  0.00282374],
        [-0.00591862,  0.00962907,  0.00478656, ...,  0.00874121,
          0.00242748,  0.00242402],
        ...,
        [ 0.00689576,  0.00703652, -0.00634631, ..., -0.00860754,
          0.00058125,  0.00429499],
        [ 0.00521316,  0.00613442, -0.00812252, ..., -0.00375527,
          0.00234178,  0.00430753],
        [-0.00412164,  0.00659016, -0.00658046, ...,  0.00275545,
          0.00525215,  0.00452137]]),
 'b1': array([ 5.62518841e-13, -3.12763154e-12, -1.92836759e-12, -1.20535930e-12,
        -1.92838056e-13,  2.25458271e-14,  1.03297481e-12,  9.84969233e-12,
        -6.13297726e-13,  5.13859284e-12, -2.57037922e-12, -6.35489182e-13,
         2.32154634e-13, -3.83276017e-12, -2.04486009e-12,  4.36790194e-12,
        -3.09324783e-15, -1.63568647e-12,  2.44616181e-12

In [8]:
import pickle

with open('affine_4.pkl', 'wb') as f:
    pickle.dump(network_dict['Affine_4'].params, f)
    

### hyper parameters
  + max_epochs = 20
  + batch_size = 100
  + iters = 5_000
  + learning_rate = 0.01

<br>

### results
+ **before image transposing** Affine_2([1000]): train_acc: 0.16316, test_acc: 0.1612, running_time: 20m
+ Affine_2[1000]: train_acc: 0.12014, test_acc: 0.1223
+ Affine_3[1000, 100]: train_acc: 0.18088, test_acc: 0.1842
+ Affine_3[1000, 500]: train_acc: 0.10366, test_acc: 0.1038
+ Affine_4[1000, 100, 50]: train_acc: 0.15788, test_acc: 0.1586
+ Affine_4[100, 100, 100]: train_acc: 0.1025, test_acc: 0.103
<br>

+ 